# Практична робота 3
**Тема:** Метод опорних векторів в системах машинного навчання
**Виконала:** Смірнова Наталія

Метод опорних векторів (SVM) — це потужний алгоритм для класифікації та регресії, що заснований на ідеї знаходження оптимальної розділяючої гіперплощини між класами. В основі лежить максимізація відстані між найближчими до площини точками кожного класу — опорними векторами.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Налаштування графіків у пастельно-рожевій палітрі з округлими елементами
sns.set_theme(style="whitegrid", rc={"axes.facecolor": "#FFF0F5", "figure.facecolor": "#FFF0F5", "axes.edgecolor": "#FFB6C1"})
plt.rcParams['font.sans-serif'] = ['Arial']


### 1. Завантаження та підготовка даних
Для реалізації алгоритму використано синтетичний датасет, який імітує мережеві атаки. Як системний аналітик, на практиці я можу підключити сюди завантажений CSV файл з Kaggle.

In [ ]:
from sklearn.datasets import make_classification

# Генерація даних (1000 зразків, 20 ознак, 2 класи)
X, y = make_classification(n_samples=1000, n_features=20, n_informative=10, 
                           n_classes=2, random_state=42)

# Розділення даних на тренувальні та тестувальні набори (80% / 20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Масштабування даних (обов'язкове для SVM)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### 2. Навчання класифікатора SVM
Класифікація полягає у перевірці, по яку сторону від гіперплощини $w^Tx+b=0$ знаходиться об'єкт.

In [ ]:
# Створення класифікатора SVM з лінійним ядром
clf_linear = SVC(kernel='linear', random_state=42)
clf_linear.fit(X_train_scaled, y_train)

# Прогнозування класів
y_pred_linear = clf_linear.predict(X_test_scaled)

# Обчислення точності
accuracy_linear = accuracy_score(y_test, y_pred_linear)
print(f"Точність (Accuracy) з лінійним ядром: {accuracy_linear:.4f}\n")
print("Звіт класифікації:")
print(classification_report(y_test, y_pred_linear))

# Матриця помилок
cm = confusion_matrix(y_test, y_pred_linear)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap=sns.light_palette("#FF69B4", as_cmap=True), 
            cbar=False, linewidths=2, linecolor='white')
plt.title('Матриця помилок (Linear SVM)', fontsize=14, fontweight='bold', color='#D2042D')
plt.xlabel('Передбачений клас', fontsize=12)
plt.ylabel('Справжній клас', fontsize=12)
plt.show()

### 3. Експеримент з нелінійним ядром (RBF)
Якщо дані лінійно нероздільні, ми використовуємо ядро $K(x,x)$, щоб перетворити їх у вищорозмірний простір.

In [ ]:
# SVM з ядром RBF
clf_rbf = SVC(kernel='rbf', C=1.0, random_state=42)
clf_rbf.fit(X_train_scaled, y_train)
y_pred_rbf = clf_rbf.predict(X_test_scaled)
print(f"Точність (Accuracy) з ядром RBF: {accuracy_score(y_test, y_pred_rbf):.4f}")

### 4. Фактори впливу на точність SVM

1. **Налаштування параметрів:** Чутливість до параметрів ядра і штрафних параметрів впливає на результат. Наприклад, штраф обмежує відхилення точок від гіперплощини.
2. **Вибір ядра (Kernel):** Для складних даних необхідно застосовувати $K(x,x)$ (RBF тощо).
3. **Масштабування ознак:** Оскільки максимізується геометрична відстань, нормалізація даних перед навчанням є обов'язковою.
4. **Висока розмірність:** Метод показує високу ефективність у багатовимірних просторах, тому добре підходить для завдань кібербезпеки.